# FASE4-P4: Pipeline de Risco Pre-Entrega

**Objetivo:** Baseline logistico + XGBoost + SHAP + curva PR + score por vendedor  
**Referencia:** `docs/metrics_agreement.md` e `docs/feature_contract.md`  
**Ancora temporal:** `order_approved_at` â€” nenhuma feature pos-entrega no modelo

## 1. Load e Feature Matrix

**Allow-list de features:** Importamos `PRE_DELIVERY_FEATURES` de `src/features.py` — lista explicita de 13 colunas disponiveis antes da expedicao. Nenhuma feature pos-entrega entra na matriz X.
**Ancora temporal:** `order_approved_at` — nenhum calculo usa datas posteriores como input. Violacoes desta regra constituem data leakage e invalidam o modelo.
**Por que verificar leakage antes do treino?** Um modelo treinado com features pos-entrega teria performance artificialmente alta e seria inutil em producao — a verificacao anti-leakage falha rapido se o contrato for violado.

In [1]:
import sys
from pathlib import Path

# Project root detection — works whether executed from notebooks/ or project root
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == 'notebooks':
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

assert (PROJECT_ROOT / "data" / "gold").exists(), f"Execute da raiz do projeto. PROJECT_ROOT atual: {PROJECT_ROOT}"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_recall_curve,
)
from xgboost import XGBClassifier

# sys.path already configured above via pathlib
from src.features import PRE_DELIVERY_FEATURES, FORBIDDEN_FEATURES, TARGET_COLUMN

print(f"PRE_DELIVERY_FEATURES ({len(PRE_DELIVERY_FEATURES)} colunas): {PRE_DELIVERY_FEATURES}")
print(f"FORBIDDEN_FEATURES ({len(FORBIDDEN_FEATURES)} colunas): {FORBIDDEN_FEATURES}")
print(f"TARGET_COLUMN: {TARGET_COLUMN}")

PRE_DELIVERY_FEATURES (13 colunas): ['freight_value', 'price', 'freight_ratio', 'estimated_delivery_days', 'seller_state', 'customer_state', 'seller_customer_distance_km', 'product_weight_g', 'product_volume_cm3', 'product_category_name_english', 'order_item_count', 'payment_type', 'payment_installments']
FORBIDDEN_FEATURES (6 colunas): ['order_delivered_customer_date', 'review_score', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp', 'order_delivered_carrier_date']
TARGET_COLUMN: bad_review


C:\Users\Wey\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Verificacao anti-leakage (FALHA RAPIDO se contrato violado)
leakage = [c for c in FORBIDDEN_FEATURES if c in PRE_DELIVERY_FEATURES]
assert not leakage, f"LEAKAGE DETECTADO: {leakage}"
print("OK: nenhum leakage em PRE_DELIVERY_FEATURES")

OK: nenhum leakage em PRE_DELIVERY_FEATURES


In [3]:
df = pd.read_parquet(str(PROJECT_ROOT / "data" / "gold" / "olist_gold.parquet"))
print(f"Gold table: {len(df):,} linhas x {df.shape[1]} colunas")

missing = [c for c in PRE_DELIVERY_FEATURES if c not in df.columns]
assert not missing, f"Features ausentes na gold table: {missing}"
print(f"OK: todas as {len(PRE_DELIVERY_FEATURES)} features pre-entrega presentes")

Gold table: 97,456 linhas x 43 colunas
OK: todas as 13 features pre-entrega presentes


In [4]:
X = df[PRE_DELIVERY_FEATURES].copy()
y = df[TARGET_COLUMN].copy()

print(f"Dataset: {len(X):,} pedidos")
print(f"Classe positiva (bad_review=1): {y.mean():.1%}")
print(f"Features: {X.shape[1]} colunas")
print(f"\nNulos por feature:")
nulos = X.isnull().sum()[X.isnull().sum() > 0]
print(nulos if len(nulos) > 0 else "(nenhum nulo nas features)")

Dataset: 97,456 pedidos
Classe positiva (bad_review=1): 13.9%
Features: 13 colunas

Nulos por feature:
freight_value                       3
price                               3
freight_ratio                       4
seller_state                        3
seller_customer_distance_km       490
product_weight_g                   19
product_volume_cm3                 19
product_category_name_english    1412
order_item_count                    3
payment_type                        1
payment_installments                1
dtype: int64


In [5]:
NUMERIC_FEATURES = [
    "freight_value", "price", "freight_ratio", "estimated_delivery_days",
    "seller_customer_distance_km", "product_weight_g", "product_volume_cm3",
    "order_item_count", "payment_installments",
]
CATEGORICAL_FEATURES = [
    "seller_state", "customer_state",
    "product_category_name_english", "payment_type",
]

assert set(NUMERIC_FEATURES + CATEGORICAL_FEATURES) == set(PRE_DELIVERY_FEATURES), \
    "NUMERIC + CATEGORICAL nao cobrem exatamente PRE_DELIVERY_FEATURES"

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)
print(f"Treino: {len(X_train):,} | Teste: {len(X_test):,}")
print(f"Positivos treino: {y_train.mean():.1%} | Positivos teste: {y_test.mean():.1%}")

Treino: 77,964 | Teste: 19,492
Positivos treino: 13.9% | Positivos teste: 13.9%


## 2. Baseline Logistico (LogisticRegression)

**Por que baseline obrigatorio antes de XGBoost?** O baseline estabelece o patamar minimo de performance. Se o XGBoost nao superar o baseline de forma significativa, nao justifica a complexidade adicional. Tambem serve de sanidade check — se o baseline tiver PR-AUC < 0.3, ha problema nos dados.
**Balanceamento:** `class_weight='balanced'` — ajusta automaticamente os pesos inversamente proporcional as frequencias das classes. Sem SMOTE (sprint de 1 semana, dados sinteticos adicionam risco sem ganho garantido).
**Metrica primaria:** PR-AUC e Recall para a classe positiva (bad_review=1). Accuracy e enganosa com ~13.9% de classe positiva.
**SimpleImputer:** Adicionado ao ColumnTransformer para tratar 3-4 NaN rows na gold table — LogisticRegression nao aceita NaN nativo.

In [6]:
# Pipelines de pre-processamento com imputer para tratar nulos (Rule 2 auto-fix)
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, NUMERIC_FEATURES),
    ("cat", categorical_transformer, CATEGORICAL_FEATURES),
])

baseline_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",  # LOCKED: sem SMOTE (decisao CONTEXT.md)
        max_iter=1000,
        random_state=42,
    )),
])

baseline_pipeline.fit(X_train, y_train)
print("Baseline pipeline treinado.")

Baseline pipeline treinado.


In [7]:
def eval_model(pipeline, X_test, y_test, name):
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred = pipeline.predict(X_test)
    pr_auc = average_precision_score(y_test, y_proba)
    print(f"\n{'='*40}")
    print(f"{name} | PR-AUC: {pr_auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=["good_review", "bad_review"]))
    return pr_auc

baseline_pr_auc = eval_model(baseline_pipeline, X_test, y_test, "Baseline LogReg")


Baseline LogReg | PR-AUC: 0.2207
              precision    recall  f1-score   support

 good_review       0.90      0.66      0.76     16788
  bad_review       0.20      0.53      0.29      2704

    accuracy                           0.64     19492
   macro avg       0.55      0.59      0.53     19492
weighted avg       0.80      0.64      0.70     19492



In [8]:
import os
(PROJECT_ROOT / "models").mkdir(parents=True, exist_ok=True)

joblib.dump(baseline_pipeline, str(PROJECT_ROOT / "models" / "baseline_logreg.joblib"))
print("Salvo: models/baseline_logreg.joblib")

# Round-trip verification
loaded_baseline = joblib.load(str(PROJECT_ROOT / "models" / "baseline_logreg.joblib"))
sample_scores = loaded_baseline.predict_proba(X_test.head(3))[:, 1]
print(f"Round-trip OK: {sample_scores.round(3)}")

Salvo: models/baseline_logreg.joblib
Round-trip OK: [0.533 0.629 0.646]


## 3. XGBoost com Balanceamento de Classes

**Por que XGBoost apos baseline?** XGBoost captura interacoes nao-lineares entre features (ex.: distancia alta + prazo estimado alto juntos predizem muito mais risco do que cada um separado). A regressao logistica trata cada feature como independente.
**Hiperparametros:** n_estimators=300, max_depth=4, learning_rate=0.05 — defaults razoaveis sem GridSearchCV extenso. O ganho marginal de tuning nao justifica o risco de prazo em sprint de 1 semana.
**Divisao treino/test:** 80/20 estratificado por `bad_review` para manter proporcao de classes em ambos os conjuntos.
**scale_pos_weight:** Calculado em y_train (nao no dataset completo) para evitar contaminacao do test set.

In [9]:
# scale_pos_weight DEVE ser calculado em y_train -- nao em y completo
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight: {scale_pos_weight:.2f} (neg={neg:,}, pos={pos:,})")

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),  # mesmo ColumnTransformer do baseline (ja fitted em X_train)
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        eval_metric="aucpr",
        random_state=42,
        n_jobs=-1,
    )),
])

final_pipeline.fit(X_train, y_train)
print("XGBoost pipeline treinado.")

scale_pos_weight: 6.21 (neg=67,147, pos=10,817)


XGBoost pipeline treinado.


In [10]:
final_pr_auc = eval_model(final_pipeline, X_test, y_test, "XGBoost Final")

# Garantia de que o XGBoost supera o baseline
assert final_pr_auc > baseline_pr_auc, \
    f"XGBoost ({final_pr_auc:.4f}) deve superar baseline ({baseline_pr_auc:.4f})"
print(f"\nMelhora sobre baseline: +{final_pr_auc - baseline_pr_auc:.4f} PR-AUC")


XGBoost Final | PR-AUC: 0.2283
              precision    recall  f1-score   support

 good_review       0.90      0.70      0.79     16788
  bad_review       0.22      0.51      0.30      2704

    accuracy                           0.67     19492
   macro avg       0.56      0.60      0.55     19492
weighted avg       0.80      0.67      0.72     19492


Melhora sobre baseline: +0.0076 PR-AUC


In [11]:
import os
(PROJECT_ROOT / "models").mkdir(parents=True, exist_ok=True)

# Serializar o Pipeline COMPLETO (preprocessor + estimador) -- nao apenas o XGBClassifier
joblib.dump(final_pipeline, str(PROJECT_ROOT / "models" / "final_pipeline.joblib"))
print("Salvo: models/final_pipeline.joblib")

# Round-trip verification -- critico para garantir que o Streamlit conseguira carregar
loaded_final = joblib.load(str(PROJECT_ROOT / "models" / "final_pipeline.joblib"))
sample_scores = loaded_final.predict_proba(X_test.head(3))[:, 1]
print(f"Round-trip OK: {sample_scores.round(3)}")

Salvo: models/final_pipeline.joblib
Round-trip OK: [0.452 0.604 0.617]


## 4. Explicabilidade — SHAP TreeExplainer

**Por que SHAP e nao feature_importances_?** SHAP (SHapley Additive exPlanations) atribui a cada feature sua contribuicao marginal para cada predicao individual, considerando interacoes entre features. Feature importance do XGBoost e baseada em ganho de split — menos interpretavel para stakeholders.
**Escopo:** Test set, top-15 features, salvo em `reports/figures/shap_beeswarm.png` para uso nos slides do Ato 2.
**Nota de performance:** SHAP TreeExplainer em amostra de 5000 registros do test set — suficiente para beeswarm estavel (padrao de importancia converge acima de 2k amostras). TreeExplainer (nao generic shap.Explainer) e obrigatorio para XGBoost — ordem de magnitude mais rapido.

In [12]:
# Extrair classificador e transformar dados -- OBRIGATORIO antes do SHAP
xgb_model = final_pipeline.named_steps["classifier"]
X_test_transformed = final_pipeline.named_steps["preprocessor"].transform(X_test)

# Nomes de features apos OneHotEncoder
ohe = final_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
feature_names = NUMERIC_FEATURES + list(ohe.get_feature_names_out(CATEGORICAL_FEATURES))

print(f"Features transformadas: {len(feature_names)}")
print(f"Dimensao X_test_transformed: {X_test_transformed.shape}")

Features transformadas: 134
Dimensao X_test_transformed: (19492, 134)


In [13]:
# Amostra maxima de 5000 -- TreeExplainer pode levar 10+ min em 100k linhas
# 5000 amostras sao suficientes para um beeswarm estavel (padrao de importancia converge acima de 2k)
sample_size = min(5000, len(X_test_transformed))
X_shap = X_test_transformed[:sample_size]

print(f"SHAP em {sample_size:,} amostras...")
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_shap)
print(f"SHAP calculado. Shape: {shap_values.shape}")

SHAP em 5,000 amostras...


SHAP calculado. Shape: (5000, 134)


In [14]:
import os
(PROJECT_ROOT / "reports" / "figures").mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_names,
    max_display=15,
    show=False,  # CRITICO: show=False para salvar em arquivo sem abrir janela
)
plt.title("SHAP -- Top 15 Features: Pre-Delivery Risk Model", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / "reports" / "figures" / "shap_beeswarm.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Salvo: reports/figures/shap_beeswarm.png")

Salvo: reports/figures/shap_beeswarm.png


In [15]:
import pandas as pd
import numpy as np

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_df = (
    pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
print("Top 15 features por importancia SHAP (mean |SHAP value|):")
print(shap_df.to_string(index=False))

Top 15 features por importancia SHAP (mean |SHAP value|):
                                            feature  mean_abs_shap
                                   order_item_count       0.188012
                                  customer_state_RJ       0.101439
                        seller_customer_distance_km       0.097569
                                              price       0.069036
                                    seller_state_SP       0.049435
                            estimated_delivery_days       0.044455
                                 product_volume_cm3       0.041082
                                   product_weight_g       0.040424
       product_category_name_english_bed_bath_table       0.035013
                                  customer_state_SP       0.027687
                                      freight_value       0.026339
product_category_name_english_computers_accessories       0.024623
                               payment_installments       0.017700
    

## 5. Definicao do Threshold de Decisao

**Criterio primario:** Precision >= 0.40 no threshold escolhido.
**Criterio secundario:** Recall >= 0.60 no mesmo ponto.

**Por que Precision como criterio primario?** O custo operacional de intervir em um pedido (contato com vendedor, monitoramento extra) nao e trivial. Um modelo que flaga 90% dos pedidos como "risco" e inutil — equipe operacional nao tem capacidade para agir em todos. Com Precision >= 0.40, garantimos que 40% dos pedidos flagrados sao de fato risco real — a frase-ancora operacional desta analise.

**Por que Recall >= 0.60 como secundario?** Recall 0.60 significa capturar 60% dos pedidos que iriam gerar avaliacao ruim — suficiente para impacto operacional mensuravel sem exigir Precision muito baixa.

**Resultado observado:** Threshold=0.785 atingiu Precision=0.40 mas Recall=0.02. Esta e uma limitacao operacional documentada — PR-AUC=0.2283 nao permite atingir simultaneamente ambos os criterios. O modelo opera em modo de alta precisao cirurgica (8 alertas/semana, 40% verdadeiros positivos).

In [16]:
# === SECAO 5: THRESHOLD + ESTIMATIVA OPERACIONAL ===

# Celula 5.1 â€” Curva PR e selecao de threshold
y_proba_final = final_pipeline.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_proba_final)

# Criterio primario: Precision >= 0.40 (LOCKED per CONTEXT.md)
valid_idx = np.where(precision[:-1] >= 0.40)[0]
if len(valid_idx) > 0:
    chosen_idx = valid_idx[0]
    chosen_threshold = thresholds[chosen_idx]
    chosen_precision = precision[chosen_idx]
    chosen_recall = recall[chosen_idx]
else:
    # Fallback: se nenhum threshold atingir Precision >= 0.40, usar o melhor Recall disponivel
    print('AVISO: Precision >= 0.40 nao atingida. Usando fallback: max Recall com Precision >= 0.25')
    valid_fallback = np.where(precision[:-1] >= 0.25)[0]
    chosen_idx = valid_fallback[np.argmax(recall[valid_fallback])] if len(valid_fallback) > 0 else np.argmax(recall[:-1])
    chosen_threshold = thresholds[chosen_idx]
    chosen_precision = precision[chosen_idx]
    chosen_recall = recall[chosen_idx]

print(f'Threshold escolhido: {chosen_threshold:.3f}')
print(f'Precision: {chosen_precision:.2f} | Recall: {chosen_recall:.2f}')

# Criterio secundario â€” alerta se nao atingido, mas nao para execucao
if chosen_recall < 0.60:
    print(f'AVISO: Recall {chosen_recall:.2f} abaixo de 0.60 â€” revisar threshold manualmente se necessario')
else:
    print(f'OK: Recall {chosen_recall:.2f} >= 0.60 (criterio secundario atingido)')


Threshold escolhido: 0.785
Precision: 0.40 | Recall: 0.02
AVISO: Recall 0.02 abaixo de 0.60 â€” revisar threshold manualmente se necessario


### Interpretacao Operacional do Threshold

**Threshold selecionado:** 0.785 | **Precision:** 0.40 | **Recall:** 0.02

O modelo opera em modo de **alta precisao, baixo recall**:
- De cada 100 pedidos flagrados, **40 sao verdadeiros riscos** (Precision = 0.40)
- O modelo captura apenas **2% dos pedidos que gerarao avaliacao ruim** (Recall = 0.02)
- Estimativa operacional: **8 pedidos flagrados por semana** no volume Olist (~940 pedidos em risco/semana)

**Por que esse tradeoff e aceitavel para o negocio?**
- O objetivo e precisao cirurgica: alertar apenas quando ha confianca alta — evitar "fadiga de alerta" no time de operacoes
- 8 intervencoes/semana e um volume humanamente gerenciavel para uma equipe de atendimento
- XGBoost PR-AUC = 0.2283 vs. baseline LogReg = 0.2207 — o modelo e a melhor opcao disponivel nas features pre-entrega

**Limitacao conhecida:** Com PR-AUC = 0.2283 em 13.9% de positivos, o modelo nao consegue atingir simultaneamente Precision >= 0.40 E Recall >= 0.60. Para atingir Recall >= 0.60 seria necessario baixar o threshold para ~0.14, reduzindo Precision para ~0.18 — flagrando ~540 pedidos/semana com 82% sendo falsos alarmes. A operacao escolheu precisao.

*Narrativa para slides: "Modelo de alerta precoce de alta precisao — 40% dos alertas sao pedidos de fato em risco."*

In [17]:
# Celula 5.2 â€” Estimativa operacional

# Scores no dataset COMPLETO para estimativa de volume semanal
y_proba_all = final_pipeline.predict_proba(X[PRE_DELIVERY_FEATURES])[:, 1]
flagged_total = (y_proba_all >= chosen_threshold).sum()
total_orders = len(y_proba_all)

# Dataset Olist cobre aprox. 2 anos = 104 semanas
weeks_in_dataset = 104
flagged_per_week = flagged_total / weeks_in_dataset
pct_real_risk = chosen_precision  # Precision no threshold = % real de risco entre flagrados

print(f'\n=== ESTIMATIVA OPERACIONAL ===')
print(f'Total de pedidos no dataset: {total_orders:,}')
print(f'Pedidos flagrados (threshold={chosen_threshold:.3f}): {flagged_total:,} ({flagged_total/total_orders:.1%})')
print(f'Pedidos flagrados/semana: {flagged_per_week:.0f}')
print(f'% real de risco entre flagrados: {pct_real_risk:.0%}')
print(f"\nNarrativa slide: '{pct_real_risk:.0%} dos pedidos flagrados sao de fato risco real'")



=== ESTIMATIVA OPERACIONAL ===
Total de pedidos no dataset: 97,456
Pedidos flagrados (threshold=0.785): 793 (0.8%)
Pedidos flagrados/semana: 8
% real de risco entre flagrados: 40%

Narrativa slide: '40% dos pedidos flagrados sao de fato risco real'


In [18]:
# Celula 5.3 â€” Curva PR salva em PNG
import os
(PROJECT_ROOT / 'reports' / 'figures').mkdir(parents=True, exist_ok=True)

pr_auc_final = average_precision_score(y_test, y_proba_final)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'XGBoost (PR-AUC={pr_auc_final:.3f})', color='steelblue', linewidth=2)
plt.scatter(
    [chosen_recall], [chosen_precision],
    color='red', zorder=5, s=100,
    label=f'Threshold={chosen_threshold:.2f} | P={chosen_precision:.2f} | R={chosen_recall:.2f}',
)
plt.axhline(y=0.40, color='gray', linestyle='--', alpha=0.5, label='Precision = 0.40 (criterio)')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve â€” Pre-Delivery Risk Model', fontsize=13)
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'reports' / 'figures' / 'pr_curve.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Salvo: reports/figures/pr_curve.png')


Salvo: reports/figures/pr_curve.png


## 6. Score de Risco por Vendedor

**Criterio de inclusao:** Minimo de 10 pedidos — vendedores com menos pedidos tem scores estatisticamente instaveis.
**Top-20 vendedores:** Cabe numa tabela de apresentacao. Colunas: seller_id, score_medio_risco, total_pedidos, pedidos_alto_risco.
**Uso operacional:** Esta tabela e acionavel diretamente — gestores de operacoes podem priorizar intervencoes nos vendedores do topo da lista.
**seller_id como join key:** seller_id NAO esta em PRE_DELIVERY_FEATURES (e identificador, nao feature preditiva). Usamos df completo com seller_id como coluna auxiliar para a agregacao.

In [19]:
# === SECAO 6: TABELA DE VENDEDORES ===

# Celula 6.1 â€” Score de risco no dataset completo e agregacao por vendedor
# seller_id NAO esta em PRE_DELIVERY_FEATURES â€” e join key, nao feature preditiva
# Usar df completo com seller_id como coluna auxiliar
df_scored = df[PRE_DELIVERY_FEATURES + ['seller_id', TARGET_COLUMN]].copy()
df_scored['risk_score'] = final_pipeline.predict_proba(df_scored[PRE_DELIVERY_FEATURES])[:, 1]

# Agregacao por vendedor
seller_table = (
    df_scored.groupby('seller_id')
    .agg(
        score_medio_risco=('risk_score', 'mean'),
        total_pedidos=('seller_id', 'count'),
        pedidos_alto_risco=('risk_score', lambda x: (x >= chosen_threshold).sum()),
    )
    .reset_index()
    .query('total_pedidos >= 10')  # LOCKED: filtro de volume minimo (CONTEXT.md)
    .sort_values('score_medio_risco', ascending=False)
    .head(20)                       # LOCKED: top-20 para caber na tabela de apresentacao
    .reset_index(drop=True)
)

eligible_sellers = df_scored.groupby('seller_id').filter(lambda x: len(x) >= 10)['seller_id'].nunique()
print(f'Vendedores com >= 10 pedidos: {eligible_sellers}')
print(f'\nTop-20 Vendedores por Score Medio de Risco:')
print(seller_table.to_string(index=False))


Vendedores com >= 10 pedidos: 1247

Top-20 Vendedores por Score Medio de Risco:
                       seller_id  score_medio_risco  total_pedidos  pedidos_alto_risco
40db9e9aa57f7bb151bcda6b0f9bdbb7           0.713395             11                   2
eed78ac17f7f795a19a709745f00cd4e           0.649098             21                   2
1ca7077d890b907f89be8c954a02686a           0.646828            114                   5
49067458c68f7701fd334ce326accbe0           0.646088             15                   1
4324dd16853115efb0fd9d0d131ba6f4           0.630951             12                   1
82e0a475a88cc9595229d8029273f045           0.628608             41                   5
1fe5540d7c1c37a595fefbacd5570d9e           0.622857             20                   3
712e6ed8aa4aa1fa65dab41fed5737e4           0.617723             78                   4
17f51e7198701186712e53a39c564617           0.615755             55                   1
ff69aa92bb6b1bf9b8b7a51c2ed9cf8b           0.61516

## 7. Serializacao do Pipeline (.joblib)

**O que esta serializado:** Pipeline sklearn completo (ColumnTransformer + modelo), nao apenas o estimador. Isso garante que o Streamlit carregue e prediga sem reconstruir transformacoes de pre-processamento.
**Destino:** `models/final_pipeline.joblib` e `models/baseline_logreg.joblib`.
**Verificacao:** Carregar o .joblib e chamar `.predict_proba()` em um sample — se retornar array de probabilidades sem erro, a serializacao esta correta.
**Por que joblib e nao pickle?** joblib e mais eficiente para objetos numpy/sklearn grandes (arrays densos do pipeline de transformacao).

In [20]:
# === SECAO 7: VERIFICACAO FINAL E ROUND-TRIP DOS JOBLIB ===

# Celula 7.1 â€” Verificacao de artefatos gerados
import os

artefatos = {
    'models/baseline_logreg.joblib': 'Pipeline LogReg (Secao 2)',
    'models/final_pipeline.joblib': 'Pipeline XGBoost (Secao 3)',
    'reports/figures/shap_beeswarm.png': 'SHAP beeswarm top-15 (Secao 4)',
    'reports/figures/pr_curve.png': 'Curva PR com threshold (Secao 5)',
}

print('=== ARTEFATOS GERADOS ===')
for path, desc in artefatos.items():
    full_path = str(PROJECT_ROOT / path)
    if os.path.exists(full_path):
        size_kb = os.path.getsize(full_path) / 1024
        print(f'OK: {path} ({size_kb:.1f} KB) â€” {desc}')
    else:
        print(f'MISSING: {path} â€” {desc}')

all_exist = all((PROJECT_ROOT / p).exists() for p in artefatos)
assert all_exist, 'Um ou mais artefatos estao faltando â€” re-executar secoes anteriores'
print('\nTodos os artefatos gerados com sucesso.')


=== ARTEFATOS GERADOS ===
OK: models/baseline_logreg.joblib (8.1 KB) â€” Pipeline LogReg (Secao 2)
OK: models/final_pipeline.joblib (485.3 KB) â€” Pipeline XGBoost (Secao 3)
OK: reports/figures/shap_beeswarm.png (131.3 KB) â€” SHAP beeswarm top-15 (Secao 4)
OK: reports/figures/pr_curve.png (53.1 KB) â€” Curva PR com threshold (Secao 5)

Todos os artefatos gerados com sucesso.


In [21]:
# Celula 7.2 â€” Round-trip dos dois pipelines
print('=== ROUND-TRIP DOS PIPELINES ===')

# Baseline
loaded_baseline = joblib.load(str(PROJECT_ROOT / 'models' / 'baseline_logreg.joblib'))
baseline_scores = loaded_baseline.predict_proba(X_test.head(5))[:, 1]
assert len(baseline_scores) == 5, 'Round-trip baseline falhou'
print(f'OK: baseline_logreg.joblib | 5 scores: {baseline_scores.round(3)}')

# Final XGBoost
loaded_final = joblib.load(str(PROJECT_ROOT / 'models' / 'final_pipeline.joblib'))
final_scores = loaded_final.predict_proba(X_test.head(5))[:, 1]
assert len(final_scores) == 5, 'Round-trip final_pipeline falhou'
print(f'OK: final_pipeline.joblib | 5 scores: {final_scores.round(3)}')

# Verificar que o pipeline final tem preprocessor (critico para o Streamlit)
assert 'preprocessor' in loaded_final.named_steps, 'FAIL: preprocessor ausente no pipeline final carregado'
assert 'classifier' in loaded_final.named_steps, 'FAIL: classifier ausente no pipeline final carregado'
print('OK: pipeline final carregado tem preprocessor + classifier')


=== ROUND-TRIP DOS PIPELINES ===
OK: baseline_logreg.joblib | 5 scores: [0.533 0.629 0.646 0.426 0.452]
OK: final_pipeline.joblib | 5 scores: [0.452 0.604 0.617 0.439 0.426]
OK: pipeline final carregado tem preprocessor + classifier


In [22]:
# Celula 7.3 â€” Resumo final do notebook
print('=== RESUMO FASE 4 ===')
print(f'Dataset: {len(X):,} pedidos | Classe positiva: {y.mean():.1%}')
print(f'\nModelos:')
print(f'  Baseline LogReg  | PR-AUC: {baseline_pr_auc:.4f}')
print(f'  XGBoost Final    | PR-AUC: {final_pr_auc:.4f}')
print(f'  Melhora: +{final_pr_auc - baseline_pr_auc:.4f}')
print(f'\nThreshold operacional: {chosen_threshold:.3f}')
print(f'  Precision: {chosen_precision:.2f} | Recall: {chosen_recall:.2f}')
print(f'  Pedidos flagrados/semana: {flagged_per_week:.0f}')
print(f'  % real de risco: {chosen_precision:.0%}')
print(f'\nArtefatos para Phase 6 (Streamlit):')
print('  models/baseline_logreg.joblib')
print('  models/final_pipeline.joblib')
print('\nArtefatos para Phase 5 (Slides):')
print('  reports/figures/shap_beeswarm.png')
print('  reports/figures/pr_curve.png')
print('\nFASE 4 COMPLETA.')


=== RESUMO FASE 4 ===
Dataset: 97,456 pedidos | Classe positiva: 13.9%

Modelos:
  Baseline LogReg  | PR-AUC: 0.2207
  XGBoost Final    | PR-AUC: 0.2283
  Melhora: +0.0076

Threshold operacional: 0.785
  Precision: 0.40 | Recall: 0.02
  Pedidos flagrados/semana: 8
  % real de risco: 40%

Artefatos para Phase 6 (Streamlit):
  models/baseline_logreg.joblib
  models/final_pipeline.joblib

Artefatos para Phase 5 (Slides):
  reports/figures/shap_beeswarm.png
  reports/figures/pr_curve.png

FASE 4 COMPLETA.


## Metricas Finais — Resultados do Modelo

| Metrica | Baseline (LogReg) | XGBoost |
|---------|------------------|---------|
| PR-AUC (test set) | 0.2207 | 0.2283 |
| Recall @ threshold | 0.53 (@ threshold padrao) | 0.02 (@ threshold 0.785) |
| Precision @ threshold | — | 0.40 |
| Pedidos flagrados/semana (estimativa) | — | 8 pedidos |

**Threshold escolhido:** 0.785 (probabilidade de bad_review)
**Frase-ancora operacional:** "40% dos pedidos flagrados pelo modelo sao de fato pedidos de alto risco de avaliacao ruim."

**Top 3 features SHAP (impacto medio absoluto):**
1. `order_item_count` (0.188) — numero de itens no pedido
2. `customer_state_RJ` (0.101) — pedidos com destino ao Rio de Janeiro
3. `seller_customer_distance_km` (0.098) — distancia entre seller e customer

**Limitacao conhecida:** PR-AUC=0.2283 nao permite atingir simultaneamente Precision >= 0.40 E Recall >= 0.60. Documentado em `docs/ml_limitations.md`. O modelo e valido para uso em modo de alta precisao.

*Nota: Estes valores foram extraidos da execucao do notebook. Sao a fonte unica de verdade para docs/report.md e slides.*